# Universal QLoRA Fine-Tuner
## Production-Grade, Multi-Dataset & Multi-Model Fine-Tuning Pipeline for Kaggle Tesla T4 GPU

This notebook runs the complete end-to-end Universal QLoRA Fine-Tuner engine.

In [ ]:
# Step 1: Environment Setup & Python Path Verification
import sys
import os
from pathlib import Path

# Determine Project Root (Local repository or Kaggle working directory)
KAGGLE_PATH = Path("/kaggle/working/Universal-QLoRA-FineTuner")
LOCAL_PATH = Path(os.getcwd())

PROJECT_ROOT = KAGGLE_PATH if KAGGLE_PATH.exists() else LOCAL_PATH
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project Root: {PROJECT_ROOT}")
print(f"Source Path:  {SRC_PATH}")

In [ ]:
# Step 2: Install QLoRA Engine Dependencies (Preserving Kaggle PyTorch/CUDA stack)
!pip install -q transformers datasets trl peft accelerate bitsandbytes sentencepiece pyyaml

In [ ]:
# Step 3: Hardware & GPU Audit
from qlora_engine.utils.hardware import detect_hardware, print_hardware_summary

hardware_info = detect_hardware()
print_hardware_summary(hardware_info)

In [ ]:
# Step 4: Load YAML Configuration
from qlora_engine.config.loader import load_config

config_path = PROJECT_ROOT / "configs" / "cybersecurity.yaml"
config = load_config(config_path)

print("Loaded Configuration:")
print(f"- Experiment : {config['project']['experiment_name']}")
print(f"- Base Model : {config['model']['name']}")
print(f"- Dataset    : {config['dataset']['name']}")

In [ ]:
# Step 5: Universal Dataset Pipeline (Loading, Schema Detection, Validation, Cleaning, Formatting, Splitting)
from qlora_engine.datasets.loader import DatasetLoader
from qlora_engine.datasets.formatter import DatasetFormatter
from qlora_engine.datasets.validator import DatasetValidator
from qlora_engine.datasets.cleaner import DatasetCleaner
from qlora_engine.datasets.splitter import DatasetSplitter

loader = DatasetLoader()
raw_ds = loader.load(config["dataset"])
print(f"Loaded raw dataset: {len(raw_ds):,} records")

schema = DatasetFormatter.detect_schema(raw_ds)
print(f"Detected Schema: '{schema}'")

valid_ds, summary = DatasetValidator.analyze_dataset(raw_ds, schema)
print("Validation Summary:", summary)

cleaned_ds = DatasetCleaner.clean(valid_ds, config["dataset"])
canonical_ds = DatasetFormatter.format_to_canonical(cleaned_ds, schema)
print(f"Canonical Dataset: {len(canonical_ds):,} records")

splits = DatasetSplitter.split(canonical_ds, seed=config["split"]["seed"])
for split_name, ds in splits.items():
    print(f"- Split '{split_name}': {len(ds):,} rows")

In [ ]:
# Step 6: Model & Tokenizer Initialization & 4-bit Quantization
from qlora_engine.models.manager import ModelManager
from qlora_engine.datasets.analyzer import TokenAnalyzer

model_mgr = ModelManager(config, hardware_info)
tokenizer = model_mgr.load_tokenizer()

# Apply chat template
formatted_splits = {}
for split_name, ds in splits.items():
    formatted_splits[split_name] = DatasetFormatter.apply_chat_template(ds, tokenizer)

token_stats = TokenAnalyzer.analyze(formatted_splits["train"], tokenizer)
print("Token Length Statistics:", token_stats)

# Load 4-bit quantized base model & attach LoRA
model = model_mgr.load_model()
model = model_mgr.attach_lora(model)

In [ ]:
# Step 7: Execute Training Smoke Test & Full Fine-Tuning Run
from qlora_engine.training.trainer import QLoraTrainer

trainer = QLoraTrainer(config, output_dir=PROJECT_ROOT / config["output"]["directory"])
training_stats = trainer.train(model, tokenizer, formatted_splits)
print("Training Run Stats:", training_stats)

In [ ]:
# Step 8: Post-Training Evaluation on Held-Out Test Split
from qlora_engine.evaluation.evaluator import ModelEvaluator

evaluator = ModelEvaluator(config)
eval_report = evaluator.evaluate(
    model=model,
    tokenizer=tokenizer,
    test_dataset=splits["test"],
    output_dir=PROJECT_ROOT / config["output"]["directory"],
    max_eval_samples=5,
)
print(f"Evaluated {eval_report['evaluated_samples_count']} test records.")

In [ ]:
# Step 9: Standalone Adapter Reload & Interactive Inference Test
from qlora_engine.inference.engine import InferenceEngine

engine = InferenceEngine(
    base_model_name=config["model"]["name"],
    adapter_path=PROJECT_ROOT / config["output"]["directory"],
)

test_prompt = "What is phishing and how can organizations prevent it?"
print(f"Test Prompt: {test_prompt}")

response = engine.generate([{"role": "user", "content": test_prompt}])
print("\n--- Fine-Tuned Model Response ---")
print(response)